# Linear Regression for Quant Finance Interviews

This notebook provides a comprehensive, mathematically rigorous treatment of linear regression --- the single most important statistical tool in quantitative finance. Every topic here has appeared in real interviews at firms like Citadel, Two Sigma, DE Shaw, Jane Street, AQR, and Goldman Sachs.

**Prerequisites:** Linear algebra (matrix operations, projections, eigenvalues), probability foundations, statistical inference (hypothesis testing, confidence intervals).

**How to use this notebook:** Work through each section carefully. The mathematical derivations are presented at the level expected in a PhD-level interview. Pay special attention to the interview tips and worked problems --- these reflect the exact types of questions you will face.

---

## 1. Ordinary Least Squares (OLS) Derivation

We begin with the foundational derivation of the OLS estimator. Understanding this from multiple perspectives --- algebraic, geometric, and probabilistic --- is essential for quant interviews.

### 1.1 The Linear Model and Normal Equations

Consider the linear model in matrix form:

$$\mathbf{y} = X\boldsymbol{\beta} + \boldsymbol{\varepsilon}$$

where $\mathbf{y} \in \mathbb{R}^n$ is the response vector, $X \in \mathbb{R}^{n \times p}$ is the design matrix (assumed to have full column rank $p$), $\boldsymbol{\beta} \in \mathbb{R}^p$ is the parameter vector, and $\boldsymbol{\varepsilon} \in \mathbb{R}^n$ is the error vector.

The OLS estimator minimizes the sum of squared residuals:

$$\hat{\boldsymbol{\beta}} = \arg\min_{\boldsymbol{\beta}} \| \mathbf{y} - X\boldsymbol{\beta} \|^2 = \arg\min_{\boldsymbol{\beta}} (\mathbf{y} - X\boldsymbol{\beta})^\top(\mathbf{y} - X\boldsymbol{\beta})$$

**Derivation via calculus.** Expand the objective:

$$S(\boldsymbol{\beta}) = \mathbf{y}^\top\mathbf{y} - 2\boldsymbol{\beta}^\top X^\top \mathbf{y} + \boldsymbol{\beta}^\top X^\top X \boldsymbol{\beta}$$

Taking the gradient and setting it to zero:

$$\frac{\partial S}{\partial \boldsymbol{\beta}} = -2X^\top \mathbf{y} + 2X^\top X \boldsymbol{\beta} = \mathbf{0}$$

This yields the **normal equations**:

$$\boxed{X^\top X \hat{\boldsymbol{\beta}} = X^\top \mathbf{y}}$$

When $X^\top X$ is invertible (full column rank), the unique solution is:

$$\boxed{\hat{\boldsymbol{\beta}} = (X^\top X)^{-1} X^\top \mathbf{y}}$$

The second-order condition confirms this is a minimum: $\frac{\partial^2 S}{\partial \boldsymbol{\beta} \partial \boldsymbol{\beta}^\top} = 2X^\top X$, which is positive definite when $X$ has full column rank.

### 1.2 Geometric Interpretation: Projection

The geometric view is arguably more important than the algebraic derivation. The column space of $X$, denoted $\mathcal{C}(X)$, is a $p$-dimensional subspace of $\mathbb{R}^n$. The fitted values $\hat{\mathbf{y}} = X\hat{\boldsymbol{\beta}}$ are the **orthogonal projection** of $\mathbf{y}$ onto $\mathcal{C}(X)$.

**Why?** The residual vector $\hat{\boldsymbol{\varepsilon}} = \mathbf{y} - \hat{\mathbf{y}}$ must be orthogonal to every column of $X$:

$$X^\top(\mathbf{y} - X\hat{\boldsymbol{\beta}}) = \mathbf{0} \quad \Longleftrightarrow \quad X^\top \hat{\boldsymbol{\varepsilon}} = \mathbf{0}$$

This is precisely the orthogonality condition that defines a projection. The residual lies in the **orthogonal complement** of $\mathcal{C}(X)$, which is $\mathcal{N}(X^\top)$ (the left null space of $X$).

**Pythagorean decomposition:**

$$\|\mathbf{y}\|^2 = \|\hat{\mathbf{y}}\|^2 + \|\hat{\boldsymbol{\varepsilon}}\|^2$$

which in statistical terms is: $\text{TSS} = \text{ESS} + \text{RSS}$ (when the model includes an intercept).

> **Interview Tip:** If asked "what does OLS do geometrically?" the answer is: OLS finds the point in the column space of $X$ that is closest to $\mathbf{y}$ in Euclidean distance. The fitted values are the orthogonal projection, and the residuals are perpendicular to every regressor. This geometric intuition is far more important than memorizing formulas.

### 1.3 The Hat Matrix

The **hat matrix** (or projection matrix) maps $\mathbf{y}$ to $\hat{\mathbf{y}}$:

$$H = X(X^\top X)^{-1}X^\top$$

so that $\hat{\mathbf{y}} = H\mathbf{y}$. The name comes from the fact that $H$ "puts the hat on $\mathbf{y}$."

**Key properties of $H$:**

1. **Idempotent:** $H^2 = H$ (projecting twice is the same as projecting once).
2. **Symmetric:** $H^\top = H$.
3. **Eigenvalues:** All eigenvalues are 0 or 1. Specifically, $H$ has $p$ eigenvalues equal to 1 and $n - p$ eigenvalues equal to 0.
4. **Trace:** $\text{tr}(H) = p$ (the number of parameters).
5. **Rank:** $\text{rank}(H) = p$.

The **residual-maker matrix** is $M = I_n - H$, which projects onto $\mathcal{C}(X)^\perp$:

$$\hat{\boldsymbol{\varepsilon}} = M\mathbf{y} = (I - H)\mathbf{y}$$

$M$ is also symmetric and idempotent, with $\text{tr}(M) = n - p$ and $\text{rank}(M) = n - p$.

**Diagonal elements** $h_{ii}$ are called **leverage values**. They satisfy $0 \leq h_{ii} \leq 1$ and $\sum_{i=1}^n h_{ii} = p$. A point with high leverage $h_{ii}$ has outsized influence on the fit.

### 1.4 Properties of OLS Residuals

Under the model $\mathbf{y} = X\boldsymbol{\beta} + \boldsymbol{\varepsilon}$ with $E[\boldsymbol{\varepsilon}] = \mathbf{0}$ and $\text{Cov}(\boldsymbol{\varepsilon}) = \sigma^2 I$:

1. **Zero mean:** $E[\hat{\boldsymbol{\varepsilon}}] = \mathbf{0}$.

2. **Orthogonality:** $X^\top \hat{\boldsymbol{\varepsilon}} = \mathbf{0}$ (residuals are orthogonal to all regressors, including the intercept if present, which implies $\sum \hat{\varepsilon}_i = 0$).

3. **Covariance matrix:** $\text{Cov}(\hat{\boldsymbol{\varepsilon}}) = \sigma^2 M = \sigma^2(I - H)$. Note that residuals are **not** independent --- they are correlated through $M$.

4. **Variance of individual residuals:** $\text{Var}(\hat{\varepsilon}_i) = \sigma^2(1 - h_{ii})$. Points with high leverage have smaller residual variance.

5. **Sum of squared residuals:** $\text{RSS} = \hat{\boldsymbol{\varepsilon}}^\top \hat{\boldsymbol{\varepsilon}} = \mathbf{y}^\top M \mathbf{y}$.

6. **Unbiased variance estimator:** $s^2 = \frac{\text{RSS}}{n - p}$ is an unbiased estimator of $\sigma^2$.

   **Proof:** $E[\text{RSS}] = E[\boldsymbol{\varepsilon}^\top M \boldsymbol{\varepsilon}] = \text{tr}(M \cdot \sigma^2 I) = \sigma^2 \text{tr}(M) = \sigma^2(n - p)$.

   The denominator $n - p$ (not $n$) accounts for the $p$ degrees of freedom used in estimation.

> **Interview Tip:** A classic question: "Why do we divide by $n - p$ instead of $n$ in the residual variance estimator?" The deep answer is that the residuals live in an $(n-p)$-dimensional subspace (the column space of $M$), so there are only $n - p$ degrees of freedom. The trace argument above makes this precise.

---

## 2. The Gauss-Markov Theorem

The Gauss-Markov theorem provides the theoretical justification for OLS. It tells us that among all linear unbiased estimators, OLS has the smallest variance.

### 2.1 Assumptions (Gauss-Markov Conditions)

The Gauss-Markov theorem requires the following assumptions:

1. **Linearity:** $\mathbf{y} = X\boldsymbol{\beta} + \boldsymbol{\varepsilon}$ (the true model is linear in parameters).
2. **Strict exogeneity:** $E[\boldsymbol{\varepsilon} \mid X] = \mathbf{0}$ (errors have zero conditional mean).
3. **Spherical errors:** $\text{Cov}(\boldsymbol{\varepsilon} \mid X) = \sigma^2 I_n$ (homoscedasticity and no autocorrelation).
4. **Full rank:** $\text{rank}(X) = p$ (no perfect multicollinearity).

**Crucially, normality of errors is NOT required.** The Gauss-Markov theorem holds for any error distribution satisfying the above conditions.

Under these assumptions, the OLS estimator has the following properties:

- **Unbiasedness:** $E[\hat{\boldsymbol{\beta}} \mid X] = \boldsymbol{\beta}$
- **Covariance:** $\text{Cov}(\hat{\boldsymbol{\beta}} \mid X) = \sigma^2(X^\top X)^{-1}$

### 2.2 BLUE: Best Linear Unbiased Estimator

**Theorem (Gauss-Markov):** Under the Gauss-Markov conditions, the OLS estimator $\hat{\boldsymbol{\beta}}$ is **BLUE** --- the Best Linear Unbiased Estimator. That is, for any linear unbiased estimator $\tilde{\boldsymbol{\beta}} = C\mathbf{y}$, we have:

$$\text{Cov}(\tilde{\boldsymbol{\beta}}) - \text{Cov}(\hat{\boldsymbol{\beta}}) \succeq 0 \quad \text{(positive semidefinite)}$$

In particular, $\text{Var}(\tilde{\beta}_j) \geq \text{Var}(\hat{\beta}_j)$ for every component $j$.

**Proof sketch:**

Let $\tilde{\boldsymbol{\beta}} = C\mathbf{y}$ be any linear unbiased estimator. Write $C = (X^\top X)^{-1}X^\top + D$ for some matrix $D$.

Unbiasedness requires $CX = I_p$, which gives:
$$[(X^\top X)^{-1}X^\top + D]X = I_p \quad \Longrightarrow \quad DX = 0$$

Now compute the covariance:

\begin{align*}
\text{Cov}(\tilde{\boldsymbol{\beta}}) &= \sigma^2 CC^\top \\
&= \sigma^2[(X^\top X)^{-1}X^\top + D][(X^\top X)^{-1}X^\top + D]^\top \\
&= \sigma^2[(X^\top X)^{-1}X^\top X(X^\top X)^{-1} + (X^\top X)^{-1}X^\top D^\top + DX(X^\top X)^{-1} + DD^\top] \\
&= \sigma^2(X^\top X)^{-1} + \sigma^2 DD^\top
\end{align*}

where we used $DX = 0$. Since $DD^\top \succeq 0$, we have:

$$\text{Cov}(\tilde{\boldsymbol{\beta}}) = \text{Cov}(\hat{\boldsymbol{\beta}}) + \sigma^2 DD^\top \succeq \text{Cov}(\hat{\boldsymbol{\beta}}) \quad \blacksquare$$

> **Interview Tip:** The Gauss-Markov theorem says OLS is the best **linear unbiased** estimator. It does NOT say OLS is the best estimator overall. Biased estimators (like ridge regression) can have lower MSE by trading a small amount of bias for a large reduction in variance. This is the bias-variance tradeoff, which is critical in quantitative finance.

### 2.3 Consequences When Assumptions Fail

| Violated Assumption | Consequence | Remedy |
|:---|:---|:---|
| Heteroscedasticity ($\text{Cov}(\varepsilon) \neq \sigma^2 I$) | OLS still unbiased but no longer efficient; standard errors are wrong | WLS, GLS, robust (White) standard errors |
| Autocorrelation | OLS still unbiased but inefficient; standard errors are wrong | GLS (Cochrane-Orcutt), Newey-West standard errors |
| Endogeneity ($E[\varepsilon \mid X] \neq 0$) | OLS is **biased and inconsistent** | Instrumental variables (2SLS), GMM |
| Multicollinearity (near-singular $X^\top X$) | OLS still BLUE but with huge variance | Ridge regression, drop variables, PCA |
| Non-linearity | Model is misspecified; OLS estimates are meaningless | Transform variables, add polynomial terms, use nonlinear models |

---

## 3. Inference in Linear Regression

For inference (hypothesis testing and confidence intervals), we add the assumption $\boldsymbol{\varepsilon} \sim N(\mathbf{0}, \sigma^2 I)$, so the full model becomes $\mathbf{y} \mid X \sim N(X\boldsymbol{\beta}, \sigma^2 I)$.

### 3.1 Distribution of the OLS Estimator

Under normality:

$$\hat{\boldsymbol{\beta}} = (X^\top X)^{-1}X^\top \mathbf{y} \sim N\big(\boldsymbol{\beta},\; \sigma^2(X^\top X)^{-1}\big)$$

since $\hat{\boldsymbol{\beta}}$ is a linear function of the normally distributed $\mathbf{y}$.

For a single coefficient $\hat{\beta}_j$:

$$\hat{\beta}_j \sim N\big(\beta_j,\; \sigma^2 [(X^\top X)^{-1}]_{jj}\big)$$

The quantity $\text{RSS}/\sigma^2 = \hat{\boldsymbol{\varepsilon}}^\top \hat{\boldsymbol{\varepsilon}}/\sigma^2 \sim \chi^2_{n-p}$, and $\hat{\boldsymbol{\beta}}$ and $s^2$ are **independent** (a consequence of Cochran's theorem and the independence of $H\mathbf{y}$ and $M\mathbf{y}$ under normality).

### 3.2 t-Tests for Individual Coefficients

To test $H_0: \beta_j = \beta_j^0$ (typically $\beta_j^0 = 0$):

$$t_j = \frac{\hat{\beta}_j - \beta_j^0}{\text{se}(\hat{\beta}_j)} \sim t_{n-p}$$

where $\text{se}(\hat{\beta}_j) = s \sqrt{[(X^\top X)^{-1}]_{jj}}$ and $s = \sqrt{\text{RSS}/(n-p)}$.

The test statistic follows a $t$-distribution with $n - p$ degrees of freedom because we replace $\sigma$ with its estimate $s$.

**$(1-\alpha)$ confidence interval for $\beta_j$:**

$$\hat{\beta}_j \pm t_{n-p, \alpha/2} \cdot \text{se}(\hat{\beta}_j)$$

### 3.3 F-Test for Overall Significance and Nested Models

**Overall F-test** ($H_0: \beta_1 = \beta_2 = \cdots = \beta_{p-1} = 0$, assuming an intercept $\beta_0$):

$$F = \frac{(\text{TSS} - \text{RSS})/({p - 1})}{\text{RSS}/(n - p)} = \frac{R^2 / (p-1)}{(1 - R^2)/(n - p)} \sim F_{p-1,\, n-p}$$

**Partial F-test** for nested models. Consider a full model with $p$ parameters and a reduced model with $q < p$ parameters:

$$F = \frac{(\text{RSS}_{\text{reduced}} - \text{RSS}_{\text{full}})/(p - q)}{\text{RSS}_{\text{full}}/(n - p)} \sim F_{p-q,\, n-p}$$

This tests whether the additional $p - q$ variables jointly contribute to the model.

**Connection between t and F:** For testing a single coefficient, $t_j^2 = F_{1, n-p}$. The $t$-test and the partial $F$-test with one restriction are equivalent.

### 3.4 Prediction Intervals vs. Confidence Intervals

For a new observation $\mathbf{x}_0$:

**Confidence interval for the mean response** $E[y_0] = \mathbf{x}_0^\top \boldsymbol{\beta}$:

$$\hat{y}_0 \pm t_{n-p, \alpha/2} \cdot s\sqrt{\mathbf{x}_0^\top (X^\top X)^{-1} \mathbf{x}_0}$$

**Prediction interval for a new observation** $y_0 = \mathbf{x}_0^\top \boldsymbol{\beta} + \varepsilon_0$:

$$\hat{y}_0 \pm t_{n-p, \alpha/2} \cdot s\sqrt{1 + \mathbf{x}_0^\top (X^\top X)^{-1} \mathbf{x}_0}$$

The prediction interval is always wider because it accounts for both estimation uncertainty and irreducible noise $\varepsilon_0$.

> **Interview Tip:** This distinction comes up frequently. The confidence interval targets the **mean** at $\mathbf{x}_0$ (estimation uncertainty only), while the prediction interval targets a **new individual observation** (estimation uncertainty plus noise). As $n \to \infty$, the confidence interval shrinks to zero width, but the prediction interval converges to $\pm z_{\alpha/2} \cdot \sigma$ --- the irreducible error floor.

### 3.5 $R^2$ and Adjusted $R^2$

**Coefficient of determination:**

$$R^2 = 1 - \frac{\text{RSS}}{\text{TSS}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

$R^2$ measures the fraction of variance explained by the model. It satisfies $0 \leq R^2 \leq 1$ (when an intercept is included) and is equal to the squared sample correlation between $y$ and $\hat{y}$.

**Problem with $R^2$:** Adding any variable (even noise) can only increase $R^2$, making it useless for model comparison with different numbers of predictors.

**Adjusted $R^2$** penalizes model complexity:

$$\bar{R}^2 = 1 - \frac{\text{RSS}/(n - p)}{\text{TSS}/(n - 1)} = 1 - \frac{n - 1}{n - p}(1 - R^2)$$

Adjusted $R^2$ can decrease when a useless variable is added, because the penalty for the additional parameter outweighs the trivial reduction in RSS.

> **Interview Tip:** In finance, $R^2$ values for cross-sectional stock return regressions are typically very low (1--5%). For time-series regressions of returns on factors, $R^2$ can be 20--70%. Never judge a model's usefulness solely by $R^2$ --- even low-$R^2$ models can generate profitable trading signals if the predictions are unbiased and the signal-to-noise ratio is exploitable through portfolio construction.

---

## 4. Multicollinearity

Multicollinearity arises when the regressors are highly correlated, making $X^\top X$ nearly singular. This is extremely common in finance, where many candidate predictors are correlated.

### 4.1 Effects on OLS Estimates

When multicollinearity is present:

- OLS estimates are still **unbiased** (Gauss-Markov still holds).
- But the variance $\sigma^2(X^\top X)^{-1}$ becomes very large because $(X^\top X)^{-1}$ has large entries.
- Coefficient estimates become **unstable** --- small changes in the data cause large swings in $\hat{\boldsymbol{\beta}}$.
- Individual coefficients may be statistically insignificant (large $p$-values) even though the overall $F$-test is significant.
- The signs of coefficients may be counterintuitive.

**Variance Inflation Factor (VIF):** For coefficient $\hat{\beta}_j$:

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ is the $R^2$ from regressing $x_j$ on all other regressors. VIF measures how much the variance of $\hat{\beta}_j$ is inflated by collinearity compared to the case of orthogonal regressors.

- $\text{VIF}_j = 1$: no collinearity with other regressors.
- $\text{VIF}_j > 5$: moderate collinearity; investigate.
- $\text{VIF}_j > 10$: severe collinearity; action needed.

**Connection to coefficient variance:**

$$\text{Var}(\hat{\beta}_j) = \frac{\sigma^2}{(n-1) \text{Var}(x_j)} \cdot \text{VIF}_j$$

### 4.2 Condition Number

The **condition number** of $X^\top X$ provides a global measure of multicollinearity:

$$\kappa(X^\top X) = \frac{\lambda_{\max}}{\lambda_{\min}}$$

where $\lambda_{\max}$ and $\lambda_{\min}$ are the largest and smallest eigenvalues of $X^\top X$.

- $\kappa < 30$: mild multicollinearity.
- $\kappa \in [30, 100]$: moderate to strong.
- $\kappa > 100$: severe multicollinearity.

High condition numbers mean that the OLS solution is numerically unstable --- small perturbations in $\mathbf{y}$ or $X$ lead to large changes in $\hat{\boldsymbol{\beta}}$.

**Remedies for multicollinearity:**

1. **Drop redundant variables** based on domain knowledge.
2. **Regularization** (ridge/lasso) --- stabilizes estimates by shrinking coefficients.
3. **Principal Component Regression (PCR)** --- regress on the top principal components of $X$.
4. **Partial Least Squares (PLS)** --- finds components that are correlated with both $X$ and $\mathbf{y}$.

> **Interview Tip:** In quant finance, factor models (e.g., Fama-French) often have correlated factors. Multicollinearity inflates the standard errors of factor loadings, making it hard to determine which factors are truly significant. Ridge regression and PCA are standard tools for handling this.

---

## 5. Regularization

Regularization introduces bias to reduce variance, often yielding lower overall prediction error. This is the most important modern extension of OLS for quantitative finance, where overfitting is the primary enemy.

### 5.1 The Bias-Variance Tradeoff

For any estimator $\hat{\boldsymbol{\beta}}$, the mean squared error decomposes as:

$$\text{MSE}(\hat{\boldsymbol{\beta}}) = E\|\hat{\boldsymbol{\beta}} - \boldsymbol{\beta}\|^2 = \underbrace{\|\text{Bias}(\hat{\boldsymbol{\beta}})\|^2}_{\text{squared bias}} + \underbrace{\text{tr}(\text{Cov}(\hat{\boldsymbol{\beta}}))}_{\text{variance}}$$

OLS has zero bias but can have very high variance (especially with multicollinearity or $p \approx n$). Regularized estimators accept some bias in exchange for a much larger reduction in variance, reducing the overall MSE.

For prediction at a new point $\mathbf{x}_0$, the expected prediction error is:

$$E[(y_0 - \hat{y}_0)^2] = \sigma^2 + \text{Bias}^2(\hat{y}_0) + \text{Var}(\hat{y}_0)$$

The first term $\sigma^2$ is irreducible noise. Regularization reduces the third term at the cost of increasing the second.

### 5.2 Ridge Regression (L2 Penalty)

Ridge regression adds an $L^2$ penalty on the coefficients:

$$\hat{\boldsymbol{\beta}}^{\text{ridge}} = \arg\min_{\boldsymbol{\beta}} \left\{ \|\mathbf{y} - X\boldsymbol{\beta}\|^2 + \lambda \|\boldsymbol{\beta}\|^2 \right\}$$

The closed-form solution is:

$$\boxed{\hat{\boldsymbol{\beta}}^{\text{ridge}} = (X^\top X + \lambda I)^{-1} X^\top \mathbf{y}}$$

**Key properties:**

1. **Always invertible:** $X^\top X + \lambda I$ is positive definite for $\lambda > 0$, even when $X^\top X$ is singular.

2. **Shrinkage toward zero:** As $\lambda \to \infty$, $\hat{\boldsymbol{\beta}}^{\text{ridge}} \to \mathbf{0}$. As $\lambda \to 0$, $\hat{\boldsymbol{\beta}}^{\text{ridge}} \to \hat{\boldsymbol{\beta}}^{\text{OLS}}$.

3. **SVD interpretation:** If $X = UDV^\top$ is the SVD, then:
$$\hat{\boldsymbol{\beta}}^{\text{ridge}} = \sum_{j=1}^p \frac{d_j^2}{d_j^2 + \lambda} \cdot \frac{\mathbf{u}_j^\top \mathbf{y}}{d_j} \mathbf{v}_j$$
   The factor $d_j^2/(d_j^2 + \lambda)$ shrinks the contribution of each principal component, with the **smallest** components (most affected by noise) shrunk the most.

4. **Bias:** $E[\hat{\boldsymbol{\beta}}^{\text{ridge}}] = (X^\top X + \lambda I)^{-1} X^\top X \boldsymbol{\beta} \neq \boldsymbol{\beta}$ (biased!).

5. **Bayesian interpretation:** Ridge regression is equivalent to the posterior mean under a Gaussian prior: $\boldsymbol{\beta} \sim N(\mathbf{0}, \tau^2 I)$ with $\lambda = \sigma^2/\tau^2$.

6. **Does not perform variable selection:** All coefficients are shrunk toward zero but none are exactly zero.

### 5.3 Lasso Regression (L1 Penalty)

The Lasso (Least Absolute Shrinkage and Selection Operator) uses an $L^1$ penalty:

$$\hat{\boldsymbol{\beta}}^{\text{lasso}} = \arg\min_{\boldsymbol{\beta}} \left\{ \frac{1}{2n}\|\mathbf{y} - X\boldsymbol{\beta}\|^2 + \lambda \|\boldsymbol{\beta}\|_1 \right\}$$

where $\|\boldsymbol{\beta}\|_1 = \sum_{j=1}^p |\beta_j|$.

**Key differences from ridge:**

1. **Sparsity:** The $L^1$ penalty drives some coefficients to **exactly zero**, performing automatic variable selection.

2. **No closed-form solution:** The $L^1$ penalty is not differentiable at zero. The solution requires iterative algorithms (coordinate descent, LARS).

3. **Geometry:** The $L^1$ constraint set $\|\boldsymbol{\beta}\|_1 \leq t$ has corners on the axes, which is why solutions tend to lie at corners (where some coordinates are zero). The $L^2$ constraint set is a smooth sphere with no corners.

4. **Soft-thresholding:** For orthogonal $X$ (i.e., $X^\top X = I$), the lasso has a closed-form solution:
$$\hat{\beta}_j^{\text{lasso}} = \text{sign}(\hat{\beta}_j^{\text{OLS}}) \max(|\hat{\beta}_j^{\text{OLS}}| - \lambda, 0)$$

5. **Bayesian interpretation:** Lasso is the posterior mode under a Laplace (double-exponential) prior: $\beta_j \sim \text{Laplace}(0, 1/\lambda)$.

### 5.4 Elastic Net

The Elastic Net combines $L^1$ and $L^2$ penalties:

$$\hat{\boldsymbol{\beta}}^{\text{enet}} = \arg\min_{\boldsymbol{\beta}} \left\{ \frac{1}{2n}\|\mathbf{y} - X\boldsymbol{\beta}\|^2 + \lambda \left( \alpha \|\boldsymbol{\beta}\|_1 + \frac{1-\alpha}{2} \|\boldsymbol{\beta}\|^2 \right) \right\}$$

where $\alpha \in [0,1]$ controls the mix ($\alpha = 1$ is lasso, $\alpha = 0$ is ridge).

**Advantages over pure lasso:**
- Handles groups of correlated variables better (lasso tends to select only one from a correlated group).
- Selects more than $n$ variables when $p > n$ (lasso can select at most $n$).
- Combines sparsity (from $L^1$) with stability (from $L^2$).

In practice, both $\lambda$ and $\alpha$ are chosen via cross-validation.

### 5.5 Cross-Validation for $\lambda$ Selection

The regularization parameter $\lambda$ is chosen by **$k$-fold cross-validation:**

1. Split the data into $k$ folds.
2. For each candidate $\lambda$ and each fold $i$, fit the model on all data except fold $i$ and compute the prediction error on fold $i$.
3. Average the prediction error across folds: $\text{CV}(\lambda) = \frac{1}{k}\sum_{i=1}^k \text{MSE}_i(\lambda)$.
4. Choose $\hat{\lambda} = \arg\min_{\lambda} \text{CV}(\lambda)$.

**Common choices:** $k = 5$ or $k = 10$. Leave-one-out CV ($k = n$) is computationally expensive but has a closed-form for ridge regression:

$$\text{LOOCV}(\lambda) = \frac{1}{n} \sum_{i=1}^n \left( \frac{y_i - \hat{y}_i(\lambda)}{1 - h_{ii}(\lambda)} \right)^2$$

where $h_{ii}(\lambda)$ is the $i$-th diagonal element of the smoother matrix $X(X^\top X + \lambda I)^{-1}X^\top$.

**The "one-standard-error rule":** Instead of choosing the $\lambda$ that minimizes CV error, choose the largest $\lambda$ whose CV error is within one standard error of the minimum. This produces a more parsimonious (simpler) model.

> **Interview Tip:** An interviewer may ask: "How do you choose the regularization parameter?" The answer is cross-validation. Be prepared to explain $k$-fold CV, discuss the bias-variance tradeoff of the choice of $k$ (small $k$ = more bias, less variance; large $k$ = less bias, more variance), and mention the one-standard-error rule as a practical heuristic.

---

## 6. Model Selection

How do we choose which variables to include? Information criteria provide principled tradeoffs between fit and complexity.

### 6.1 Information Criteria

All information criteria have the form: **Goodness of fit + Complexity penalty**.

**Akaike Information Criterion (AIC):**

$$\text{AIC} = -2 \ln \hat{L} + 2p = n \ln(\text{RSS}/n) + 2p + \text{const}$$

AIC is derived as an estimate of the expected Kullback-Leibler divergence from the true model. It is asymptotically equivalent to leave-one-out cross-validation.

**Bayesian Information Criterion (BIC):**

$$\text{BIC} = -2 \ln \hat{L} + p \ln n = n \ln(\text{RSS}/n) + p \ln n + \text{const}$$

BIC penalizes complexity more heavily than AIC (since $\ln n > 2$ for $n \geq 8$). BIC is consistent: it selects the true model with probability approaching 1 as $n \to \infty$. AIC tends to overfit.

**Mallow's $C_p$:**

$$C_p = \frac{\text{RSS}_p}{\hat{\sigma}^2_{\text{full}}} - n + 2p$$

where $\hat{\sigma}^2_{\text{full}}$ is the variance estimate from the full model. For the correct model, $E[C_p] \approx p$. Choose the model with $C_p$ closest to $p$.

| Criterion | Penalty | Tends to... | Best for... |
|:---|:---|:---|:---|
| AIC | $2p$ | Select larger models | Prediction |
| BIC | $p \ln n$ | Select smaller models | Identifying the "true" model |
| $C_p$ | $2p$ | Equivalent to AIC (under normality) | Classical setting |

> **Interview Tip:** If asked "AIC vs BIC --- when do you use which?" the key distinction is: AIC optimizes prediction accuracy (minimizes KL divergence), while BIC optimizes model identification (consistent selection of the true model). In quant finance, prediction is usually the goal, so AIC or cross-validation is preferred.

### 6.2 Stepwise Selection and Cross-Validation

**Forward selection:** Start with no variables; at each step, add the variable that most reduces the RSS (or equivalently, has the largest partial $F$-statistic). Stop when no addition is significant.

**Backward elimination:** Start with all variables; at each step, remove the variable with the smallest partial $F$-statistic (least significant). Stop when all remaining variables are significant.

**Stepwise (bidirectional):** Combine forward and backward at each step.

**Problems with stepwise selection:**
- Not guaranteed to find the globally best subset.
- $p$-values are invalid because of sequential testing (multiple comparisons not corrected).
- Overly optimistic $R^2$ on the selected model.

**Cross-validation** avoids these problems: evaluate each candidate model by its out-of-sample prediction error. This is the gold standard for model selection in quant finance, where the ultimate test is whether a model predicts well on unseen data.

---

## 7. Diagnostics

Model diagnostics verify whether the assumptions underlying OLS are satisfied and identify problematic observations.

### 7.1 Residual Analysis

**Types of residuals:**

1. **Raw residuals:** $\hat{\varepsilon}_i = y_i - \hat{y}_i$.

2. **Standardized residuals:** $r_i = \hat{\varepsilon}_i / (s\sqrt{1 - h_{ii}})$. These have approximately unit variance.

3. **Studentized (externally studentized) residuals:** $t_i = \hat{\varepsilon}_i / (s_{(i)}\sqrt{1 - h_{ii}})$ where $s_{(i)}$ is the residual standard error computed **without** observation $i$. Under the null hypothesis (no outliers), $t_i \sim t_{n-p-1}$.

**Key diagnostic plots:**

- **Residuals vs. fitted values:** Should show no pattern. A funnel shape indicates heteroscedasticity; a curve indicates non-linearity.
- **Normal Q-Q plot of residuals:** Points should fall along a straight line. Deviations indicate non-normality (heavy tails, skewness).
- **Residuals vs. each predictor:** Check for missed nonlinear relationships.
- **Scale-location plot** ($\sqrt{|r_i|}$ vs. fitted values): Check for heteroscedasticity.

### 7.2 Leverage and Influence

**Leverage** $h_{ii}$ measures how far observation $i$ is from the center of the predictor space. High-leverage points have $h_{ii} > 2p/n$.

**Influence** measures how much removing an observation changes the fit. High leverage combined with a large residual produces high influence.

**Cook's distance** combines leverage and residual size:

$$D_i = \frac{r_i^2}{p} \cdot \frac{h_{ii}}{1 - h_{ii}} = \frac{(\hat{\mathbf{y}} - \hat{\mathbf{y}}_{(i)})^\top(\hat{\mathbf{y}} - \hat{\mathbf{y}}_{(i)})}{p \cdot s^2}$$

where $\hat{\mathbf{y}}_{(i)}$ is the vector of fitted values when observation $i$ is deleted.

- $D_i > 0.5$: observation $i$ is worth investigating.
- $D_i > 1$: observation $i$ is highly influential.

**DFFITS** measures the influence on the fitted value at point $i$:

$$\text{DFFITS}_i = t_i \sqrt{\frac{h_{ii}}{1 - h_{ii}}}$$

Threshold: $|\text{DFFITS}_i| > 2\sqrt{p/n}$.

> **Interview Tip:** In quant finance, a few extreme observations (market crashes, earnings surprises) can dominate regression results. Always check for influential points. A robust regression approach that downweights outliers may be more appropriate than removing them entirely.

### 7.3 Tests for Heteroscedasticity

**Breusch-Pagan test:**

1. Run the original OLS regression and obtain residuals $\hat{\varepsilon}_i$.
2. Regress $\hat{\varepsilon}_i^2$ on the original regressors $X$.
3. The test statistic is $\text{BP} = nR^2_{\text{aux}} \sim \chi^2_p$ under $H_0$ (homoscedasticity).

**White test:**

A more general version that does not assume a specific form for heteroscedasticity:

1. Regress $\hat{\varepsilon}_i^2$ on the original regressors, their squares, and all cross-products.
2. $\text{White} = nR^2_{\text{aux}} \sim \chi^2_q$ where $q$ is the number of regressors in the auxiliary regression.

**If heteroscedasticity is detected:**
- Use **White (heteroscedasticity-consistent) standard errors**: $\widehat{\text{Cov}}(\hat{\boldsymbol{\beta}}) = (X^\top X)^{-1} X^\top \text{diag}(\hat{\varepsilon}_i^2) X (X^\top X)^{-1}$.
- Or use **Weighted Least Squares (WLS)** if the variance function is known.

### 7.4 Tests for Autocorrelation

**Durbin-Watson test** for first-order autocorrelation $\text{AR}(1)$:

$$\text{DW} = \frac{\sum_{i=2}^n (\hat{\varepsilon}_i - \hat{\varepsilon}_{i-1})^2}{\sum_{i=1}^n \hat{\varepsilon}_i^2} \approx 2(1 - \hat{\rho})$$

where $\hat{\rho}$ is the sample first-order autocorrelation of residuals.

- $\text{DW} \approx 2$: no autocorrelation.
- $\text{DW} \ll 2$ (close to 0): positive autocorrelation.
- $\text{DW} \gg 2$ (close to 4): negative autocorrelation.

**If autocorrelation is present:**
- Use **Newey-West standard errors** (HAC --- heteroscedasticity and autocorrelation consistent).
- Or use **Generalized Least Squares (GLS)** with an appropriate error structure.

> **Interview Tip:** In time-series finance applications, autocorrelated residuals are the norm rather than the exception. Always use Newey-West standard errors when running time-series regressions. The number of lags is typically chosen as $\lfloor 4(T/100)^{2/9} \rfloor$ or based on the frequency of the data (e.g., 12 lags for monthly data with annual effects).

---

## 8. Generalized Linear Models (GLM)

GLMs extend linear regression to response variables that are not normally distributed. The key idea is to model a transformation of the mean through a **link function**.

### 8.1 GLM Framework

A GLM has three components:

1. **Random component:** $Y_i$ follows an exponential family distribution: $f(y_i; \theta_i, \phi) = \exp\left\{\frac{y_i \theta_i - b(\theta_i)}{a(\phi)} + c(y_i, \phi)\right\}$

   where $\theta_i$ is the canonical parameter and $\phi$ is the dispersion parameter.

2. **Systematic component:** $\eta_i = \mathbf{x}_i^\top \boldsymbol{\beta}$ (the linear predictor).

3. **Link function:** $g(\mu_i) = \eta_i$ where $\mu_i = E[Y_i]$.

| Distribution | Canonical Link | $g(\mu)$ | $\mu = g^{-1}(\eta)$ |
|:---|:---|:---|:---|
| Normal | Identity | $\mu$ | $\eta$ |
| Bernoulli | Logit | $\ln\frac{\mu}{1-\mu}$ | $\frac{e^\eta}{1+e^\eta}$ |
| Poisson | Log | $\ln \mu$ | $e^\eta$ |
| Gamma | Inverse | $1/\mu$ | $1/\eta$ |

**Key properties of exponential families:**
- $E[Y_i] = \mu_i = b'(\theta_i)$
- $\text{Var}(Y_i) = a(\phi) \cdot b''(\theta_i) = a(\phi) \cdot V(\mu_i)$

where $V(\mu)$ is the **variance function**.

### 8.2 Logistic Regression

For binary outcomes $Y_i \in \{0, 1\}$, logistic regression models:

$$P(Y_i = 1 \mid \mathbf{x}_i) = \pi_i = \frac{\exp(\mathbf{x}_i^\top \boldsymbol{\beta})}{1 + \exp(\mathbf{x}_i^\top \boldsymbol{\beta})} = \sigma(\mathbf{x}_i^\top \boldsymbol{\beta})$$

where $\sigma(z) = 1/(1 + e^{-z})$ is the sigmoid function. Equivalently:

$$\ln \frac{\pi_i}{1 - \pi_i} = \mathbf{x}_i^\top \boldsymbol{\beta}$$

The left side is the **log-odds** (logit). The coefficients $\beta_j$ represent the change in log-odds for a one-unit increase in $x_j$, holding other variables constant.

**Estimation** is by maximum likelihood. The log-likelihood is:

$$\ell(\boldsymbol{\beta}) = \sum_{i=1}^n \left[ y_i \ln \pi_i + (1 - y_i) \ln(1 - \pi_i) \right] = \sum_{i=1}^n \left[ y_i \mathbf{x}_i^\top \boldsymbol{\beta} - \ln(1 + e^{\mathbf{x}_i^\top \boldsymbol{\beta}}) \right]$$

This is concave, so the MLE is unique (if it exists). There is no closed-form solution; estimation uses iteratively reweighted least squares (IRLS) or Newton-Raphson.

**Interpretation of coefficients:** $e^{\beta_j}$ is the **odds ratio** --- the multiplicative change in odds for a one-unit increase in $x_j$.

> **Interview Tip:** Logistic regression is used extensively in quant finance for classification tasks: predicting default/no default, up/down market moves, or trade execution success. When asked about logistic regression in an interview, emphasize the log-odds interpretation and the fact that the loss function (cross-entropy) is convex.

### 8.3 Poisson Regression

For count data $Y_i \in \{0, 1, 2, \ldots\}$, Poisson regression models:

$$Y_i \sim \text{Poisson}(\mu_i), \qquad \ln \mu_i = \mathbf{x}_i^\top \boldsymbol{\beta}$$

so $\mu_i = \exp(\mathbf{x}_i^\top \boldsymbol{\beta})$. The log link ensures $\mu_i > 0$.

**Key feature:** The Poisson model assumes $\text{Var}(Y_i) = E[Y_i] = \mu_i$ (equidispersion). In practice, count data often exhibits **overdispersion** ($\text{Var}(Y_i) > \mu_i$), requiring:

- **Quasi-Poisson:** Uses the Poisson model but inflates standard errors by an estimated dispersion parameter.
- **Negative Binomial regression:** $Y_i \sim \text{NegBin}(r, p_i)$, which allows $\text{Var}(Y_i) = \mu_i + \mu_i^2/r > \mu_i$.

**Finance application:** Poisson regression is used to model event counts --- number of trades in an interval, number of defaults in a credit portfolio, or number of earnings surprises.

---

## 9. Finance Applications

This section covers the direct applications of linear regression to quantitative finance. These topics are heavily tested in interviews at systematic hedge funds and asset management firms.

### 9.1 Factor Models and Fama-French

The **single-factor model (CAPM)** states that the excess return of asset $i$ is:

$$R_{i,t} - R_{f,t} = \alpha_i + \beta_i (R_{m,t} - R_{f,t}) + \varepsilon_{i,t}$$

- $\alpha_i$: the asset's "alpha" (abnormal return). Under the CAPM, $\alpha_i = 0$ for all assets.
- $\beta_i$: the asset's market sensitivity (systematic risk loading).
- $\varepsilon_{i,t}$: idiosyncratic return (diversifiable risk).

The **Fama-French 3-factor model** extends this:

$$R_{i,t} - R_{f,t} = \alpha_i + \beta_i^{\text{MKT}} \text{MKT}_t + \beta_i^{\text{SMB}} \text{SMB}_t + \beta_i^{\text{HML}} \text{HML}_t + \varepsilon_{i,t}$$

where:
- $\text{MKT}_t = R_{m,t} - R_{f,t}$ (market excess return)
- $\text{SMB}_t$ (Small Minus Big): return spread between small-cap and large-cap stocks
- $\text{HML}_t$ (High Minus Low): return spread between value and growth stocks

Modern extensions include the **Fama-French 5-factor model** (adding RMW for profitability and CMA for investment) and the **Carhart 4-factor model** (adding UMD for momentum).

**Estimating beta:** Run a time-series OLS regression of the asset's excess returns on the factor returns. The slope coefficient is $\hat{\beta}_i$. Use Newey-West standard errors to account for autocorrelation.

> **Interview Tip:** Alpha is the holy grail of quantitative investing. A statistically significant positive $\alpha$ means the asset or strategy earns returns beyond what its factor exposures explain. In practice, apparent alpha often disappears once you account for more factors, transaction costs, or data-snooping bias.

### 9.2 Alpha Estimation and Testing

Given a strategy's return time series $\{R_t\}_{t=1}^T$, estimate alpha by regressing excess returns on risk factors:

$$R_t - R_{f,t} = \alpha + \sum_{k=1}^K \beta_k F_{k,t} + \varepsilon_t$$

**Testing $H_0: \alpha = 0$:**

$$t_{\alpha} = \frac{\hat{\alpha}}{\text{se}(\hat{\alpha})} \sim t_{T - K - 1}$$

Using Newey-West standard errors for robustness.

**Information ratio** (annualized):

$$\text{IR} = \frac{\hat{\alpha}}{\hat{\sigma}_{\varepsilon}} \cdot \sqrt{\text{periods per year}}$$

The IR measures the alpha per unit of idiosyncratic risk. An IR above 0.5 is considered good; above 1.0 is exceptional.

**GRS test** (Gibbons, Ross, Shanken, 1989): Tests whether all alphas are jointly zero across $N$ assets:

$$\text{GRS} = \frac{T - N - K}{N} \cdot \frac{\hat{\boldsymbol{\alpha}}^\top \hat{\Sigma}^{-1} \hat{\boldsymbol{\alpha}}}{1 + \bar{F}^\top \hat{\Omega}_F^{-1} \bar{F}} \sim F_{N, T-N-K}$$

This is the multivariate generalization of the individual alpha $t$-test.

### 9.3 Cross-Sectional Regression: Fama-MacBeth

The **Fama-MacBeth (1973) procedure** estimates factor risk premia from cross-sectional data. It is the standard methodology for testing asset pricing models.

**Step 1: Time-series regressions.** For each asset $i = 1, \ldots, N$, run:

$$R_{i,t} = \alpha_i + \sum_{k=1}^K \beta_{i,k} F_{k,t} + \varepsilon_{i,t}, \quad t = 1, \ldots, T$$

to obtain estimated betas $\hat{\beta}_{i,k}$.

**Step 2: Cross-sectional regressions.** For each time period $t$, run:

$$R_{i,t} = \gamma_{0,t} + \sum_{k=1}^K \gamma_{k,t} \hat{\beta}_{i,k} + u_{i,t}, \quad i = 1, \ldots, N$$

This gives time-varying estimates of factor risk premia $\hat{\gamma}_{k,t}$.

**Step 3: Average and test.** Compute:

$$\hat{\gamma}_k = \frac{1}{T} \sum_{t=1}^T \hat{\gamma}_{k,t}, \qquad \text{se}(\hat{\gamma}_k) = \frac{1}{\sqrt{T}} \text{sd}(\hat{\gamma}_{k,t})$$

Test $H_0: \gamma_k = 0$ using a $t$-test. The Shanken (1992) correction adjusts standard errors for the errors-in-variables problem (estimated betas used as regressors).

**Advantages of Fama-MacBeth:**
- Naturally handles cross-sectional correlation (by averaging across time).
- Allows time-varying risk premia.
- Straightforward to implement.

> **Interview Tip:** The Fama-MacBeth procedure is one of the most commonly asked topics in quant research interviews. Be prepared to explain each step, discuss the errors-in-variables problem (betas are estimated, not observed), and know the Shanken correction. Also be aware that the procedure assumes no time-series dependence in the cross-sectional errors.

### 9.4 Signal Construction and Backtesting Alphas

In quantitative equity investing, the typical workflow is:

1. **Signal construction:** Create a cross-sectional signal (e.g., momentum score, value score, earnings surprise) for each stock at each time period.

2. **Predictive regression:** Test whether the signal predicts future returns:
$$R_{i,t+1} = \alpha + \gamma \cdot \text{Signal}_{i,t} + \boldsymbol{\beta}^\top \text{Controls}_{i,t} + \varepsilon_{i,t+1}$$

   A statistically significant $\hat{\gamma}$ suggests the signal has predictive power.

3. **Portfolio sorts:** Sort stocks into quantiles (e.g., deciles) based on the signal. Form long-short portfolios (long top decile, short bottom decile). Evaluate performance:
   - Average return spread.
   - Sharpe ratio of the long-short portfolio.
   - Alpha after controlling for known factors.
   - Monotonicity of returns across quantiles.

4. **Backtesting considerations:**
   - **Look-ahead bias:** Never use information that was not available at the time of the trading decision.
   - **Survivorship bias:** Include delisted stocks to avoid overstating returns.
   - **Transaction costs:** Account for bid-ask spreads, market impact, and turnover.
   - **Data-snooping:** Multiple testing inflates the probability of finding spurious signals. Use techniques like the Bonferroni correction, False Discovery Rate (FDR), or out-of-sample validation.

**Regression with Newey-West standard errors** is the standard for testing signal predictiveness. In cross-sectional settings, use Fama-MacBeth regressions or clustered standard errors (clustered by time and/or by firm).

---

## 10. Classic Interview Problems

The following problems represent the types of regression questions that commonly appear in quant finance interviews. Work through each one carefully.

### Problem 1: Deriving the OLS Estimator in Simple Linear Regression

> **Problem:** For the model $y_i = \beta_0 + \beta_1 x_i + \varepsilon_i$, derive the OLS estimators $\hat{\beta}_0$ and $\hat{\beta}_1$ from first principles. Show that $\hat{\beta}_1 = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sum(x_i - \bar{x})^2}$.

**Solution:**

Minimize $S(\beta_0, \beta_1) = \sum_{i=1}^n (y_i - \beta_0 - \beta_1 x_i)^2$.

First-order conditions:

\begin{align*}
\frac{\partial S}{\partial \beta_0} &= -2\sum_{i=1}^n (y_i - \beta_0 - \beta_1 x_i) = 0 \\
\frac{\partial S}{\partial \beta_1} &= -2\sum_{i=1}^n x_i(y_i - \beta_0 - \beta_1 x_i) = 0
\end{align*}

From the first equation:
$$\sum y_i = n\beta_0 + \beta_1 \sum x_i \quad \Longrightarrow \quad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$$

Substituting into the second equation:
$$\sum x_i(y_i - \bar{y} + \hat{\beta}_1 \bar{x} - \hat{\beta}_1 x_i) = 0$$
$$\sum x_i(y_i - \bar{y}) = \hat{\beta}_1 \sum x_i(x_i - \bar{x})$$

Since $\sum x_i(y_i - \bar{y}) = \sum(x_i - \bar{x})(y_i - \bar{y})$ (centering does not change cross-products when summed) and $\sum x_i(x_i - \bar{x}) = \sum(x_i - \bar{x})^2$:

$$\boxed{\hat{\beta}_1 = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^n (x_i - \bar{x})^2} = \frac{S_{xy}}{S_{xx}}} \qquad \blacksquare$$

Note that $\hat{\beta}_1 = r_{xy} \cdot (s_y / s_x)$ where $r_{xy}$ is the sample correlation.

### Problem 2: Omitted Variable Bias

> **Problem:** Suppose the true model is $y = X_1 \beta_1 + X_2 \beta_2 + \varepsilon$, but you estimate the short regression $y = X_1 \tilde{\beta}_1 + \tilde{\varepsilon}$. Derive the bias of $\tilde{\beta}_1$ and explain when it is zero.

**Solution:**

The OLS estimator from the short regression is:

$$\tilde{\beta}_1 = (X_1^\top X_1)^{-1}X_1^\top \mathbf{y} = (X_1^\top X_1)^{-1}X_1^\top(X_1\beta_1 + X_2\beta_2 + \varepsilon)$$

$$= \beta_1 + \underbrace{(X_1^\top X_1)^{-1}X_1^\top X_2}_{\delta}\beta_2 + (X_1^\top X_1)^{-1}X_1^\top \varepsilon$$

Taking expectations (using $E[\varepsilon \mid X] = 0$):

$$E[\tilde{\beta}_1] = \beta_1 + \delta \beta_2$$

$$\boxed{\text{Bias}(\tilde{\beta}_1) = \delta \beta_2 = (X_1^\top X_1)^{-1}X_1^\top X_2 \cdot \beta_2}$$

Note that $\delta = (X_1^\top X_1)^{-1}X_1^\top X_2$ is the matrix of regression coefficients from regressing $X_2$ on $X_1$.

**The bias is zero when:**
1. $\beta_2 = 0$ (the omitted variable is irrelevant), **OR**
2. $\delta = 0$ (the included and omitted variables are orthogonal, i.e., $X_1^\top X_2 = 0$).

**Direction of bias (scalar case):**

$$\text{Bias}(\tilde{\beta}_1) = \delta \cdot \beta_2 = \text{(correlation of } X_1 \text{ and } X_2) \times \text{(effect of } X_2)$$

| | $\beta_2 > 0$ | $\beta_2 < 0$ |
|:---|:---|:---|
| $\delta > 0$ | Positive bias (overestimate) | Negative bias (underestimate) |
| $\delta < 0$ | Negative bias | Positive bias |

> **Interview Tip:** Omitted variable bias is one of the most frequently tested concepts in econometrics interviews. The formula $\text{bias} = \delta \cdot \beta_2$ encodes the two conditions for bias: the omitted variable must both (i) affect the outcome and (ii) be correlated with the included variables. In finance, omitting a risk factor leads to biased alpha estimates.

### Problem 3: Properties of $R^2$ Under Transformations

> **Problem:** You run a regression $y = \beta_0 + \beta_1 x + \varepsilon$ and get $R^2 = 0.64$. If you now regress $x$ on $y$ (reversing the roles), what is the $R^2$ of this reversed regression?

**Solution:**

In simple linear regression, $R^2$ equals the squared sample correlation between $x$ and $y$:

$$R^2 = r_{xy}^2$$

Since correlation is symmetric ($r_{xy} = r_{yx}$), the $R^2$ from regressing $x$ on $y$ is:

$$R^2_{\text{reversed}} = r_{yx}^2 = r_{xy}^2 = R^2 = \boxed{0.64}$$

The $R^2$ is the same for both regressions. However, the **slope coefficients are different:**

- Forward: $\hat{\beta}_1 = r_{xy}(s_y/s_x)$
- Reversed: $\hat{\gamma}_1 = r_{xy}(s_x/s_y)$

Note that $\hat{\beta}_1 \cdot \hat{\gamma}_1 = r_{xy}^2 = R^2$, and $\hat{\gamma}_1 \neq 1/\hat{\beta}_1$ in general (unless $R^2 = 1$).

**Important caveat:** This $R^2$ symmetry only holds for simple linear regression (one predictor). In multiple regression, $R^2$ is not symmetric.

### Problem 4: Ridge Regression and Eigenvalue Shrinkage

> **Problem:** Show that ridge regression shrinks the OLS solution along the principal component directions, with greater shrinkage for directions with smaller eigenvalues. What is the effective degrees of freedom of ridge regression?

**Solution:**

Let $X = UDV^\top$ be the SVD, where $D = \text{diag}(d_1, \ldots, d_p)$ with $d_1 \geq \cdots \geq d_p > 0$. Then $X^\top X = VD^2V^\top$.

The OLS solution is:
$$\hat{\boldsymbol{\beta}}^{\text{OLS}} = VD^{-1}U^\top \mathbf{y} = \sum_{j=1}^p \frac{\mathbf{u}_j^\top \mathbf{y}}{d_j} \mathbf{v}_j$$

The ridge solution is:
$$\hat{\boldsymbol{\beta}}^{\text{ridge}} = V(D^2 + \lambda I)^{-1}DU^\top \mathbf{y} = \sum_{j=1}^p \frac{d_j^2}{d_j^2 + \lambda} \cdot \frac{\mathbf{u}_j^\top \mathbf{y}}{d_j} \mathbf{v}_j$$

Each principal component direction $\mathbf{v}_j$ is multiplied by the **shrinkage factor** $\frac{d_j^2}{d_j^2 + \lambda} \in (0, 1)$.

For large singular values ($d_j^2 \gg \lambda$), the shrinkage factor $\approx 1$ (little shrinkage).
For small singular values ($d_j^2 \ll \lambda$), the shrinkage factor $\approx 0$ (aggressive shrinkage).

This is exactly the right behavior: directions with small eigenvalues are poorly estimated (high variance), so they should be shrunk more.

**Effective degrees of freedom:**

$$\text{df}(\lambda) = \text{tr}(H_{\lambda}) = \text{tr}\left(X(X^\top X + \lambda I)^{-1}X^\top\right) = \sum_{j=1}^p \frac{d_j^2}{d_j^2 + \lambda}$$

When $\lambda = 0$: $\text{df} = p$ (full OLS). As $\lambda \to \infty$: $\text{df} \to 0$. The effective degrees of freedom interpolates smoothly between 0 and $p$, providing a continuous measure of model complexity. $\quad \blacksquare$

### Problem 5: Fama-MacBeth Regression

> **Problem:** You have monthly returns for 500 stocks over 120 months. You want to test whether a momentum signal predicts cross-sectional returns after controlling for market beta and size. Walk through the Fama-MacBeth procedure. What is the errors-in-variables problem, and how does the Shanken correction address it?

**Solution:**

**Step 1: Estimate betas.** For each stock $i$, run a time-series regression over a rolling window (e.g., 60 months):
$$R_{i,t} - R_{f,t} = \alpha_i + \beta_i (R_{m,t} - R_{f,t}) + \varepsilon_{i,t}$$
Obtain $\hat{\beta}_i$ and size measure $\text{Size}_{i,t} = \ln(\text{MarketCap}_{i,t})$.

**Step 2: Monthly cross-sectional regressions.** For each month $t$:
$$R_{i,t} = \gamma_{0,t} + \gamma_{1,t} \hat{\beta}_i + \gamma_{2,t} \text{Size}_{i,t-1} + \gamma_{3,t} \text{Mom}_{i,t-1} + u_{i,t}$$
Note: use lagged characteristics to avoid look-ahead bias.

**Step 3: Average and test.** Compute $\hat{\gamma}_3 = \frac{1}{T}\sum_{t=1}^T \hat{\gamma}_{3,t}$ and $\text{se}(\hat{\gamma}_3) = \text{sd}(\hat{\gamma}_{3,t})/\sqrt{T}$.

Test $H_0: \gamma_3 = 0$ (momentum has no cross-sectional predictive power) with $t = \hat{\gamma}_3/\text{se}(\hat{\gamma}_3)$.

**Errors-in-variables problem:** In Step 2, we use $\hat{\beta}_i$ instead of the true $\beta_i$. The measurement error $\hat{\beta}_i - \beta_i$ attenuates the cross-sectional slope $\gamma_1$ toward zero and inflates its standard error.

**Shanken (1992) correction:** The corrected covariance matrix is:

$$\text{Cov}^{\text{Shanken}}(\hat{\boldsymbol{\gamma}}) = (1 + c) \cdot \text{Cov}^{\text{FM}}(\hat{\boldsymbol{\gamma}}) + \frac{c}{T} \hat{\Sigma}_{\text{cs}}$$

where $c = \bar{F}^\top \hat{\Omega}_F^{-1} \bar{F}$ and $\hat{\Sigma}_{\text{cs}}$ is the covariance of the cross-sectional regression errors. The factor $(1 + c)$ inflates standard errors to account for the estimation uncertainty in betas.

### Problem 6: Heteroscedasticity and Robust Standard Errors

> **Problem:** You estimate a CAPM regression for a stock and obtain $\hat{\beta} = 1.2$ with a standard error of 0.15 using classical OLS. Using White robust standard errors, the standard error becomes 0.25. (a) Is $\hat{\beta}$ still valid? (b) Which standard error should you use? (c) What does this imply about the distribution of residuals?

**Solution:**

**(a)** Yes, $\hat{\beta} = 1.2$ is still a valid (unbiased, consistent) estimate of $\beta$. OLS point estimates remain valid under heteroscedasticity --- only the standard errors are affected. This follows from the Gauss-Markov result: unbiasedness requires $E[\varepsilon \mid X] = 0$, not homoscedasticity.

**(b)** Use the White robust standard error of 0.25. The classical standard error of 0.15 assumes homoscedasticity, which appears to be violated. Using the classical SE would lead to:
- Overstated $t$-statistics: $t_{\text{classical}} = 1.2/0.15 = 8.0$ vs. $t_{\text{robust}} = 1.2/0.25 = 4.8$.
- Understated $p$-values.
- Confidence intervals that are too narrow.

In this case, both are significant, but the robust SE is 67% larger, indicating substantial heteroscedasticity.

**(c)** The fact that $\text{se}_{\text{robust}} > \text{se}_{\text{classical}}$ implies that the residual variance is larger for some observations (likely periods of high market volatility) and smaller for others. In finance, this is common: stock return volatility is time-varying (ARCH/GARCH effects). The residuals likely exhibit **volatility clustering** --- large residuals tend to follow large residuals.

$$\text{White SE:} \quad \widehat{\text{Var}}(\hat{\beta}) = (X^\top X)^{-1}\left(\sum_{i=1}^n \hat{\varepsilon}_i^2 \mathbf{x}_i \mathbf{x}_i^\top\right)(X^\top X)^{-1}$$

This is the "sandwich" estimator: bread = $(X^\top X)^{-1}$, meat = $\sum \hat{\varepsilon}_i^2 \mathbf{x}_i \mathbf{x}_i^\top$.

> **Interview Tip:** In practice, always report robust standard errors for financial regressions. It costs you nothing when homoscedasticity holds (robust SEs are consistent either way) and protects you when it fails. Some interviewers will specifically ask why the classical and robust SEs differ and what it implies.

---

## Summary: Key Formulas Quick Reference

| Topic | Formula |
|:---|:---|
| OLS estimator | $\hat{\boldsymbol{\beta}} = (X^\top X)^{-1}X^\top \mathbf{y}$ |
| Hat matrix | $H = X(X^\top X)^{-1}X^\top$, $\text{tr}(H) = p$ |
| OLS variance | $\text{Cov}(\hat{\boldsymbol{\beta}}) = \sigma^2(X^\top X)^{-1}$ |
| Residual variance estimator | $s^2 = \text{RSS}/(n-p)$ |
| $t$-test for $\beta_j$ | $t_j = \hat{\beta}_j / \text{se}(\hat{\beta}_j) \sim t_{n-p}$ |
| $F$-test (overall) | $F = \frac{R^2/(p-1)}{(1-R^2)/(n-p)} \sim F_{p-1, n-p}$ |
| $R^2$ | $1 - \text{RSS}/\text{TSS}$ |
| Adjusted $R^2$ | $1 - \frac{n-1}{n-p}(1-R^2)$ |
| VIF | $\text{VIF}_j = 1/(1 - R_j^2)$ |
| Ridge estimator | $(X^\top X + \lambda I)^{-1}X^\top \mathbf{y}$ |
| Ridge effective df | $\sum_j d_j^2/(d_j^2 + \lambda)$ |
| Lasso (orthogonal) | $\text{sign}(\hat{\beta}_j^{\text{OLS}})\max(|\hat{\beta}_j^{\text{OLS}}| - \lambda, 0)$ |
| AIC | $n\ln(\text{RSS}/n) + 2p$ |
| BIC | $n\ln(\text{RSS}/n) + p\ln n$ |
| Cook's distance | $D_i = \frac{r_i^2}{p} \cdot \frac{h_{ii}}{1-h_{ii}}$ |
| Durbin-Watson | $\text{DW} \approx 2(1 - \hat{\rho})$ |
| Logistic regression | $\ln\frac{\pi}{1-\pi} = \mathbf{x}^\top\boldsymbol{\beta}$ |
| Omitted variable bias | $\text{Bias} = (X_1^\top X_1)^{-1}X_1^\top X_2 \cdot \beta_2$ |
| White robust SE | $(X^\top X)^{-1}(\sum \hat{\varepsilon}_i^2 \mathbf{x}_i \mathbf{x}_i^\top)(X^\top X)^{-1}$ |

---

## Further Reading

- **Greene**, *Econometric Analysis* --- the standard graduate econometrics reference, covering everything from OLS to GMM.
- **Hastie, Tibshirani, & Friedman**, *The Elements of Statistical Learning* --- the bible for regularization, model selection, and statistical learning. Freely available online.
- **Cochrane**, *Asset Pricing* --- the definitive text on factor models, cross-sectional regressions, and the Fama-MacBeth procedure.
- **Angrist & Pischke**, *Mostly Harmless Econometrics* --- accessible treatment of causal inference, omitted variable bias, and instrumental variables.
- **Joshi**, *Quant Job Interview Questions and Answers* --- regression and statistics problems curated for quant interviews.
- **Bali, Engle, & Murray**, *Empirical Asset Pricing* --- hands-on guide to implementing cross-sectional tests and factor models.